# 동료 작업 연동

이전 단계들에서 수집하고 분석한 데이터를 바탕으로 동료의 작업(Neo4j 업데이트 등)을 실행합니다.

## Parameters

In [ ]:
# Papermill parameters (DAG에서 전달받음)
env = "stg"
dt = "2024-08-07" 
version_date = "20240807"

# 노트북 내부 설정
min_changes_threshold = 1
log_level = "INFO"

## 1. 라이브러리 및 설정 로드

In [ ]:
import json
import logging
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Dict, Any, Optional
import sys
import os

# 로깅 설정
logging.basicConfig(level=getattr(logging, log_level.upper()))
logger = logging.getLogger(__name__)

print(f"환경: {env}")
print(f"처리 날짜: {dt}")
print(f"버전 날짜: {version_date}")
print(f"변경사항 임계값: {min_changes_threshold}")

## 2. 변경사항 분석 및 처리 조건 확인

In [ ]:
    # 최소 변경사항 임계값 설정
    if change_summary.get('total_new_field_changes', 0) < min_changes_threshold:
        should_process = False
        print(f"\n⚠️  변경사항이 임계값({min_changes_threshold})보다 적어서 동료 작업을 건너뜁니다.")
    
    process_decision = {
        "should_process": should_process,
        "reason": "변경사항이 임계값 미만" if not should_process else "처리 조건 충족",
        "threshold_check": {
            "min_required": min_changes_threshold,
            "actual_changes": change_summary.get('total_new_field_changes', 0)
        }
    }

## 3. 동료 노트북 실행 준비

In [ ]:
colleague_execution_result = None

if should_process:
    print(f"\n=== 동료 노트북 실행 준비 ===")
    
    # 동료 노트북에 전달할 파라미터 준비
    colleague_parameters = {
        "input_data_file": saved_file_path,
        "version_date": version_date,
        "changes_summary": change_summary,
        "has_new_products": change_summary.get('completely_new_products', 0) > 0,
        "has_field_changes": change_summary.get('products_with_new_fields', 0) > 0,
        "total_changes": change_summary.get('total_new_field_changes', 0)
    }
    
    print(f"동료에게 전달할 파라미터:")
    for key, value in colleague_parameters.items():
        print(f"- {key}: {value}")
    
    # 동료 노트북 파일 존재 확인
    colleague_path = Path(colleague_notebook_path)
    
    if colleague_path.exists():
        print(f"\n✅ 동료 노트북 파일 확인: {colleague_path}")
        notebook_ready = True
    else:
        print(f"\n❌ 동료 노트북 파일을 찾을 수 없습니다: {colleague_path}")
        print(f"   현재는 Mock 실행으로 진행합니다.")
        notebook_ready = False
        
else:
    print(f"\n처리 조건을 만족하지 않아 동료 작업을 건너뜁니다.")
    notebook_ready = False

## 4. 동료 작업 실행

In [ ]:
if should_process:
    print(f"\n=== 동료 작업 실행 ===")
    
    execution_start_time = datetime.now()
    
    if notebook_ready:
        # 실제 Papermill로 동료 노트북 실행
        try:
            import papermill as pm
            
            # 실행 결과를 저장할 경로
            output_notebook_path = f"/tmp/colleague_executed_{version_date}_{datetime.now().strftime('%H%M%S')}.ipynb"
            
            print(f"Papermill로 동료 노트북 실행 중...")
            print(f"입력: {colleague_notebook_path}")
            print(f"출력: {output_notebook_path}")
            
            # Papermill 실행
            pm.execute_notebook(
                input_path=colleague_notebook_path,
                output_path=output_notebook_path,
                parameters=colleague_parameters,
                kernel_name="python3"
            )
            
            execution_end_time = datetime.now()
            execution_duration = (execution_end_time - execution_start_time).total_seconds()
            
            colleague_execution_result = {
                "status": "success",
                "execution_time": execution_duration,
                "output_notebook": output_notebook_path,
                "parameters_sent": colleague_parameters,
                "timestamp": execution_end_time.isoformat()
            }
            
            print(f"\n✅ 동료 노트북 실행 완료!")
            print(f"실행 시간: {execution_duration:.2f}초")
            print(f"결과 노트북: {output_notebook_path}")
            
        except ImportError:
            print(f"\n❌ Papermill이 설치되지 않았습니다. Mock 실행으로 진행합니다.")
            notebook_ready = False
        except Exception as e:
            print(f"\n❌ 동료 노트북 실행 실패: {str(e)}")
            
            colleague_execution_result = {
                "status": "failed",
                "error": str(e),
                "parameters_sent": colleague_parameters,
                "timestamp": datetime.now().isoformat()
            }
    
    if not notebook_ready:
        # Mock 실행 (개발/테스트용)
        print(f"\n🔧 Mock 동료 작업 실행")
        print(f"실제 환경에서는 다음 작업들이 수행될 예정입니다:")
        print(f"1. Neo4j 데이터베이스에 새 상품 정보 업로드")
        print(f"2. 기존 상품의 새 필드 정보 업데이트")
        print(f"3. 인덱스 및 관계 재구성")
        print(f"4. 데이터 무결성 검증")
        
        # Mock 처리 시간 시뮬레이션
        import time
        mock_processing_time = min(10, change_summary.get('total_new_field_changes', 0) * 0.1)
        print(f"\nMock 처리 중... (예상 시간: {mock_processing_time:.1f}초)")
        time.sleep(mock_processing_time)
        
        execution_end_time = datetime.now()
        execution_duration = (execution_end_time - execution_start_time).total_seconds()
        
        colleague_execution_result = {
            "status": "mock_success",
            "execution_time": execution_duration,
            "mock_processing": True,
            "parameters_prepared": colleague_parameters,
            "timestamp": execution_end_time.isoformat(),
            "note": "실제 동료 노트북이 실행되지 않았습니다. Mock 처리됨."
        }
        
        print(f"✅ Mock 처리 완료 (소요시간: {execution_duration:.2f}초)")

else:
    print(f"\n⏭️  처리 조건 미충족으로 동료 작업을 건너뜁니다.")
    colleague_execution_result = {
        "status": "skipped",
        "reason": process_decision["reason"],
        "timestamp": datetime.now().isoformat()
    }

## 5. 결과 정리 및 최종 보고

In [ ]:
# 전체 파이프라인 실행 결과 정리
final_pipeline_result = {
    "pipeline_completion_time": datetime.now().isoformat(),
    "version_date": version_date,
    "stages": {
        "data_collection": {
            "status": "completed",
            "output_file": saved_file_path
        },
        "change_detection": {
            "status": "completed",
            "has_changes": has_changes,
            "summary": change_summary,
            "output_files": change_detection_result.get('output_files', [])
        },
        "colleague_processing": colleague_execution_result
    },
    "overall_status": "success" if colleague_execution_result.get("status") in ["success", "mock_success", "skipped"] else "partial_success"
}

print(f"\n=== 전체 파이프라인 실행 완료 ===")
print(f"버전: {version_date}")
print(f"전체 상태: {final_pipeline_result['overall_status']}")

print(f"\n단계별 결과:")
print(f"1. 데이터 수집: ✅ 완료 - {saved_file_path}")

print(f"2. 변경사항 감지: ✅ 완료")
if has_changes:
    print(f"   → 변경사항 {change_summary.get('total_new_field_changes', 0)}개 감지")
    print(f"   → 저장된 분석 파일 {len(change_detection_result.get('output_files', []))}개")
else:
    print(f"   → 변경사항 없음")

colleague_status = colleague_execution_result.get("status", "unknown")
if colleague_status == "success":
    print(f"3. 동료 작업: ✅ 성공 - 실행시간 {colleague_execution_result.get('execution_time', 0):.2f}초")
elif colleague_status == "mock_success":
    print(f"3. 동료 작업: 🔧 Mock 성공 - 실행시간 {colleague_execution_result.get('execution_time', 0):.2f}초")
elif colleague_status == "skipped":
    print(f"3. 동료 작업: ⏭️  건너뜀 - {colleague_execution_result.get('reason', 'Unknown')}")
elif colleague_status == "failed":
    print(f"3. 동료 작업: ❌ 실패 - {colleague_execution_result.get('error', 'Unknown error')}")

# 최종 결과를 다음 시스템에서 활용할 수 있도록 변수에 저장
pipeline_final_result = final_pipeline_result

print(f"\n파이프라인 실행 완료. 결과가 pipeline_final_result 변수에 저장되었습니다.")

# 중요한 메트릭 출력
if has_changes:
    print(f"\n📊 주요 지표:")
    print(f"- 새 상품: {change_summary.get('completely_new_products', 0)}개")
    print(f"- 필드 변경된 상품: {change_summary.get('products_with_new_fields', 0)}개")
    print(f"- 총 변경사항: {change_summary.get('total_new_field_changes', 0)}개")
    print(f"- 동료 작업 상태: {colleague_status}")

## 완료

전체 데이터 동기화 파이프라인이 완료되었습니다.

### 실행된 단계
1. **Product ID 수집**: MAP API에서 활성 상품 ID 리스트 수집
2. **상세 정보 수집**: 각 상품의 상세 정보를 병렬로 수집하여 JSON 파일로 저장
3. **변경사항 감지**: 이전 데이터와 비교하여 새로 추가된 필드만 감지
4. **동료 작업 실행**: 변경사항이 있는 경우 동료의 후속 작업(Neo4j 업데이트 등) 실행

### 출력 파일
- 상품 데이터: `mobile_plan_info_YYYYMMDD.json`
- 변경사항 분석: `field_changes_YYYYMMDD.json`, `field_changes_summary_YYYYMMDD.txt`

### 다음 실행
다음 번 실행 시에는 오늘 수집한 데이터가 이전 데이터로 사용되어 새로운 변경사항을 감지합니다.